# Familiarisation Session Analysis

Same psychophysics tasks and measures as `analysis_behavioural.ipynb` (cut-off frequency,
contrast sensitivity, interocular imbalance, dichoptic noise, stereoacuity, crowded/uncrowded
acuity), but for the "familiarisation" stage - the practice session participants complete
before pretraining, used here as an exploratory look at the data rather than a pre/post
training comparison.

In [ ]:
import importlib
import import_packages # all my packages are there, it's easier
importlib.reload(import_packages) # so I can add new packages in import_packages.py without restarting the kernel
from import_packages import *
print("Packages (re)loaded successfully!")

In [ ]:
# Set Up the paths 
ori = os.getcwd()

path_data_ref = os.path.abspath(os.path.join(ori, "..", "Data", "Refined"))
path_plot = os.path.abspath(os.path.join(ori, "..", "Data", "Plots"))

In [ ]:
bv_thresholds = pd.read_excel(path_data_ref+'/bv_2_results_thresholds.xlsx', sheet_name=None) 
bv_diff = pd.read_excel(path_data_ref+'/bv_3_results_Diff.xlsx', sheet_name=None) 

# Cut-off
Highest spatial frequency (cycles/degree) at which a grating can still be detected.

In [ ]:
df = bv_diff['cutoff'].copy()


In [ ]:
# cutoff - test
df = bv_diff['cutoff'].copy()
df['group'] = 'test'
indexID = df[(df['ID'] == 'P01')].index
df.drop(indexID, inplace=True)

df['amb'] = df['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_long = df.melt(id_vars=['ID', 'amb', 'group'], value_vars=['familiarisation','pretraining', 'posttraining'],
                  var_name='Stage', value_name='Frequency')
df_long['Stage'] = df_long['Stage'].map({'familiarisation': 'Familiarisation', 'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})




In [ ]:
df_long_clean = df_long.copy()
df_long_clean_BE = df_long_clean[(df_long_clean['amb'] == 'Both Eyes')]
df_long_clean_BE['amb'] = pd.Categorical(df_long_clean_BE['amb'], categories=['Both Eyes'], ordered=True)

df_long_clean = df_long_clean[(df_long_clean['amb'] != 'Both Eyes')]
df_long_clean['amb'] = pd.Categorical(df_long_clean['amb'], categories=['Strong Eye', 'Amblyopic Eye'], ordered=True)


In [ ]:


F_cutoff = df_long_clean.copy()
# for the amblyopes
aov = pg.rm_anova(dv='Frequency', within=['amb','Stage'], subject='ID', data=df_long_clean)
aov.round(3)

In [ ]:


###################################################################################

# Set a better style
sns.set_theme(style="white", palette="pastel")

# Set up the plot
plt.figure(figsize=(7, 6))

# Create the boxplot with colors for each amb condition
ax = sns.boxplot(
    x='amb', y='Frequency', data=df_long_clean, hue='Stage', showfliers=False, width=0.6, linewidth=1.5, 
    #palette={'Lazy Eye': '#66CDAA', 'Both Eyes': '#1E90FF', 'Strong Eye': '#FF6347'}
    palette={'Familiarisation': "#0063D5",'Pre-Training': '#D55E00','Post-Training': '#009E73' }
)

ax.axhline(y=22.1, color='blue', linestyle='--', linewidth=3, zorder=1, alpha=0.2)

# strip = sns.stripplot(data=df_long_clean, x="amb", y="Frequency", hue='Stage', dodge=True, ax=ax, 
#                 palette={'Familiarisation': "#0063D5",'Pre-Training': '#D55E00','Post-training': '#009E73' }, edgecolor='black', linewidth=1)

ax.set_yticks(range(0, 35, 5))
#ax.set_yticklabels([5, 10, 15, 20, 25])

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=[(handles[0]), (handles[1]), (handles[2])],
          labels=['Familiarisation', 'Pre-Training', 'Post-Training'],
          loc='lower right', handlelength=3,
          handler_map={tuple: HandlerTuple(ndivide=None)})

# Map amb categories to numerical positions for plotting
amb_order = df_long_clean['amb'].unique()
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# Customize axis labels
ax.set_ylabel("Frequency (cycles per dva)", fontsize=14)  # or whatever label you'd like
ax.set_xlabel("", fontsize=14)  # removes the label but keeps 'Strong Eye' / 'Lazy Eye' ticks

# Get cleaned categorical order for amb
amb_order = df_long_clean['amb'].cat.categories
amb_mapping = {val: i for i, val in enumerate(amb_order)}

import numpy as np
import matplotlib.lines as mlines

# --- Assumptions / set these to match your data ---
stages = ['Familiarisation', 'Pre-Training', 'Post-Training']  # exact names in df_long_clean['Stage']
palette = {'Familiarisation': "#0063D5",'Pre-Training': '#D55E00','Post-Training': '#009E73' }

# --- ensure amb is categorical and ordered (you already did this) ---
amb_order = df_long_clean['amb'].cat.categories
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# --- choose how wide the dodge should be (tweak if points overlap or too far) ---
# This is a fraction of the box width; typical values ~0.18-0.30
dodge_width = 0.24
# compute offsets centered around 0 for N stages
n_stages = len(stages)
if n_stages == 1:
    stage_offsets = {stages[0]: 0.0}
else:
    offsets_array = np.linspace(-dodge_width, dodge_width, n_stages)
    stage_offsets = {st: offsets_array[i] for i, st in enumerate(stages)}

# --- jitter amount for scatter so points don't exactly overlap vertically/horizontally ---
horiz_jitter = 0.02   # tiny horizontal jitter to make overlapping subjects visible
vert_jitter = 0.0     # usually don't jitter vertical, keep real y

# --- draw scatter points using deterministic x positions (do NOT rely on stripplot offsets) ---
for st in stages:
    sub = df_long_clean[df_long_clean['Stage'] == st]
    if sub.empty:
        continue
    x_vals = [amb_mapping[a] + stage_offsets[st] + np.random.uniform(-horiz_jitter, horiz_jitter)
              for a in sub['amb']]
    y_vals = sub['Frequency'].values
    ax.scatter(x_vals, y_vals,
               label=st,
               edgecolor='black', linewidth=1,
               s=30, alpha=0.9,
               zorder=3,
               marker='o',
               facecolor=palette.get(st, 'gray'))

# --- build coord_map from df values (deterministic) so we can draw subject lines consistently ---
coord_map = {}  # {(ID, amb): {stage: (x, y)}}
for _, row in df_long_clean.iterrows():
    key = (row['ID'], row['amb'])
    st = row['Stage']
    if st not in stage_offsets:
        continue
    x = amb_mapping[row['amb']] + stage_offsets[st]
    y = row['Frequency']
    coord_map.setdefault(key, {})[st] = (x, y)

# --- plot subject-wise lines across stages (only if present) ---
for key, d in coord_map.items():
    # Choose order of stages to draw line through:
    x_line = []
    y_line = []
    for st in stages:
        if st in d:
            x_line.append(d[st][0])
            y_line.append(d[st][1])
    if len(x_line) >= 2:
        ax.plot(x_line, y_line, color='black', alpha=0.4, linewidth=1, zorder=2)

# --- tidy legend: keep only one entry per stage and put in lower right like before ---
# create custom handles to ensure consistent order
handles = []
labels = []
for st in stages:
    handles.append(mlines.Line2D([], [], color=palette[st], marker='o', linestyle='None',
                                 markeredgecolor='black', markersize=6))
    labels.append(st)

ax.legend(handles=handles, labels=labels, loc='lower left', handlelength=3)

# Example of adding a significance bar
x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
y, h, col = 31, 1, 'black'
ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
ax.text((x1+x2)*.5, y+h+0.2, "***", ha='center', va='bottom', color=col)

# ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [28, 28+h, 28+h, 28], lw=1.5, c=col)
# ax.text((x1+x1)*.5, 28+h+0.2, "n.s.", ha='center', va='bottom', color=col)

# ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [26, 26+h, 26+h, 26], lw=1.5, c=col)
# ax.text((x2+x2)*.5, 26+h+0.2, "n.s.", ha='center', va='bottom', color=col)
plt.title('Acuity across sessions in amblyopes', fontsize = 16)
plt.savefig('../Data/Plots/Narrative_plots/fam/cutoff.png', dpi=600, bbox_inches='tight')
plt.figure(figsize=(8, 6))





In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# filter for one eye
df_eye = df_long_clean[df_long_clean['amb'] == 'Amblyopic Eye']

# pivot to wide format
df_wide = df_eye.pivot(index='ID', columns='Stage', values='Frequency')

# choose comparison
pre = df_wide['Familiarisation']
post = df_wide['Pre-Training']

mean_vals = (pre + post) / 2
diff_vals = post - pre

bias = np.mean(diff_vals)
loa_upper = bias + 1.96*np.std(diff_vals, ddof=1)
loa_lower = bias - 1.96*np.std(diff_vals, ddof=1)

plt.figure(figsize=(6,5))
plt.scatter(mean_vals, diff_vals)
plt.axhline(bias, linestyle='--')
plt.axhline(loa_upper, linestyle=':')
plt.axhline(loa_lower, linestyle=':')
plt.xlabel("Mean of Fam & Pre")
plt.ylabel("Pre - Fam")
plt.title("Bland-Altman (Amblyopic Eye)")
plt.show()

print("Bias:", bias)
print("Upper LoA:", loa_upper)
print("Lower LoA:", loa_lower)

################################################################################################

# choose comparison
pre = df_wide['Pre-Training']
post = df_wide['Post-Training']

mean_vals = (pre + post) / 2
diff_vals = post - pre

bias = np.mean(diff_vals)
loa_upper = bias + 1.96*np.std(diff_vals, ddof=1)
loa_lower = bias - 1.96*np.std(diff_vals, ddof=1)

plt.figure(figsize=(6,5))
plt.scatter(mean_vals, diff_vals)
plt.axhline(bias, linestyle='--')
plt.axhline(loa_upper, linestyle=':')
plt.axhline(loa_lower, linestyle=':')
plt.xlabel("Mean of Pre & Post")
plt.ylabel("Post - Pre")
plt.title("Bland-Altman (Amblyopic Eye)")
plt.show()

print("Bias:", bias)
print("Upper LoA:", loa_upper)
print("Lower LoA:", loa_lower)


################################################################################################

# choose comparison
pre = df_wide['Familiarisation']
post = df_wide['Post-Training']

mean_vals = (pre + post) / 2
diff_vals = post - pre

bias = np.mean(diff_vals)
loa_upper = bias + 1.96*np.std(diff_vals, ddof=1)
loa_lower = bias - 1.96*np.std(diff_vals, ddof=1)

plt.figure(figsize=(6,5))
plt.scatter(mean_vals, diff_vals)
plt.axhline(bias, linestyle='--')
plt.axhline(loa_upper, linestyle=':')
plt.axhline(loa_lower, linestyle=':')
plt.xlabel("Mean of Fam & Post")
plt.ylabel("Post - Fam")
plt.title("Bland-Altman (Amblyopic Eye)")
plt.show()

print("Bias:", bias)
print("Upper LoA:", loa_upper)
print("Lower LoA:", loa_lower)


##############################################################################
##############################################################################

# filter for one eye
df_eye = df_long_clean[df_long_clean['amb'] == 'Strong Eye']

# pivot to wide format
df_wide = df_eye.pivot(index='ID', columns='Stage', values='Frequency')

# choose comparison
pre = df_wide['Familiarisation']
post = df_wide['Pre-Training']

mean_vals = (pre + post) / 2
diff_vals = post - pre

bias = np.mean(diff_vals)
loa_upper = bias + 1.96*np.std(diff_vals, ddof=1)
loa_lower = bias - 1.96*np.std(diff_vals, ddof=1)

plt.figure(figsize=(6,5))
plt.scatter(mean_vals, diff_vals)
plt.axhline(bias, linestyle='--')
plt.axhline(loa_upper, linestyle=':')
plt.axhline(loa_lower, linestyle=':')
plt.xlabel("Mean of Fam & Pre")
plt.ylabel("Pre - Fam")
plt.title("Bland-Altman (Strong Eye)")
plt.show()

print("Bias:", bias)
print("Upper LoA:", loa_upper)
print("Lower LoA:", loa_lower)

################################################################################################

# choose comparison
pre = df_wide['Pre-Training']
post = df_wide['Post-Training']

mean_vals = (pre + post) / 2
diff_vals = post - pre

bias = np.mean(diff_vals)
loa_upper = bias + 1.96*np.std(diff_vals, ddof=1)
loa_lower = bias - 1.96*np.std(diff_vals, ddof=1)

plt.figure(figsize=(6,5))
plt.scatter(mean_vals, diff_vals)
plt.axhline(bias, linestyle='--')
plt.axhline(loa_upper, linestyle=':')
plt.axhline(loa_lower, linestyle=':')
plt.xlabel("Mean of Pre & Post")
plt.ylabel("Post - Pre")
plt.title("Bland-Altman (Strong Eye)")
plt.show()

print("Bias:", bias)
print("Upper LoA:", loa_upper)
print("Lower LoA:", loa_lower)


################################################################################################

# choose comparison
pre = df_wide['Familiarisation']
post = df_wide['Post-Training']

mean_vals = (pre + post) / 2
diff_vals = post - pre

bias = np.mean(diff_vals)
loa_upper = bias + 1.96*np.std(diff_vals, ddof=1)
loa_lower = bias - 1.96*np.std(diff_vals, ddof=1)

plt.figure(figsize=(6,5))
plt.scatter(mean_vals, diff_vals)
plt.axhline(bias, linestyle='--')
plt.axhline(loa_upper, linestyle=':')
plt.axhline(loa_lower, linestyle=':')
plt.xlabel("Mean of Fam & Post")
plt.ylabel("Post - Fam")
plt.title("Bland-Altman (Strong Eye)")
plt.show()

print("Bias:", bias)
print("Upper LoA:", loa_upper)
print("Lower LoA:", loa_lower)



In [ ]:


###################################################################################

# Set a better style
sns.set_theme(style="white", palette="pastel")

# Set up the plot
plt.figure(figsize=(7, 6))

# Create the boxplot with colors for each amb condition
ax = sns.boxplot(
    x='amb', y='Frequency', data=df_long_clean_BE, hue='Stage', showfliers=False, width=0.6, linewidth=1.5, 
    #palette={'Lazy Eye': '#66CDAA', 'Both Eyes': '#1E90FF', 'Strong Eye': '#FF6347'}
    palette={'Familiarisation': "#0063D5",'Pre-Training': '#D55E00','Post-Training': '#009E73' }
)

ax.axhline(y=22.1, color='blue', linestyle='--', linewidth=3, zorder=1, alpha=0.2)

# strip = sns.stripplot(data=df_long_clean, x="amb", y="Frequency", hue='Stage', dodge=True, ax=ax, 
#                 palette={'Familiarisation': "#0063D5",'Pre-Training': '#D55E00','Post-training': '#009E73' }, edgecolor='black', linewidth=1)

ax.set_yticks(range(0, 35, 5))
#ax.set_yticklabels([5, 10, 15, 20, 25])

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=[(handles[0]), (handles[1]), (handles[2])],
          labels=['Familiarisation', 'Pre-Training', 'Post-Training'],
          loc='lower right', handlelength=3,
          handler_map={tuple: HandlerTuple(ndivide=None)})

# Map amb categories to numerical positions for plotting
amb_order = df_long_clean_BE['amb'].unique()
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# Customize axis labels
ax.set_ylabel("Frequency (cycles per dva)", fontsize=14)  # or whatever label you'd like
ax.set_xlabel("")  # removes the label but keeps 'Strong Eye' / 'Lazy Eye' ticks

# Get cleaned categorical order for amb
amb_order = df_long_clean_BE['amb'].cat.categories
amb_mapping = {val: i for i, val in enumerate(amb_order)}

import numpy as np
import matplotlib.lines as mlines

# --- Assumptions / set these to match your data ---
stages = ['Familiarisation', 'Pre-Training', 'Post-Training']  # exact names in df_long_clean['Stage']
palette = {'Familiarisation': "#0063D5",'Pre-Training': '#D55E00','Post-Training': '#009E73' }

# --- ensure amb is categorical and ordered (you already did this) ---
amb_order = df_long_clean_BE['amb'].cat.categories
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# --- choose how wide the dodge should be (tweak if points overlap or too far) ---
# This is a fraction of the box width; typical values ~0.18-0.30
dodge_width = 0.24
# compute offsets centered around 0 for N stages
n_stages = len(stages)
if n_stages == 1:
    stage_offsets = {stages[0]: 0.0}
else:
    offsets_array = np.linspace(-dodge_width, dodge_width, n_stages)
    stage_offsets = {st: offsets_array[i] for i, st in enumerate(stages)}

# --- jitter amount for scatter so points don't exactly overlap vertically/horizontally ---
horiz_jitter = 0.0   # tiny horizontal jitter to make overlapping subjects visible
vert_jitter = 0.0     # usually don't jitter vertical, keep real y

# --- draw scatter points using deterministic x positions (do NOT rely on stripplot offsets) ---
for st in stages:
    sub = df_long_clean_BE[df_long_clean_BE['Stage'] == st]
    if sub.empty:
        continue
    x_vals = [amb_mapping[a] + stage_offsets[st] + np.random.uniform(-horiz_jitter, horiz_jitter)
              for a in sub['amb']]
    y_vals = sub['Frequency'].values
    ax.scatter(x_vals, y_vals,
               label=st,
               edgecolor='black', linewidth=1,
               s=30, alpha=0.9,
               zorder=3,
               marker='o',
               facecolor=palette.get(st, 'gray'))

# --- build coord_map from df values (deterministic) so we can draw subject lines consistently ---
coord_map = {}  # {(ID, amb): {stage: (x, y)}}
for _, row in df_long_clean_BE.iterrows():
    key = (row['ID'], row['amb'])
    st = row['Stage']
    if st not in stage_offsets:
        continue
    x = amb_mapping[row['amb']] + stage_offsets[st]
    y = row['Frequency']
    coord_map.setdefault(key, {})[st] = (x, y)

# --- plot subject-wise lines across stages (only if present) ---
for key, d in coord_map.items():
    # Choose order of stages to draw line through:
    x_line = []
    y_line = []
    for st in stages:
        if st in d:
            x_line.append(d[st][0])
            y_line.append(d[st][1])
    if len(x_line) >= 2:
        ax.plot(x_line, y_line, color='black', alpha=0.4, linewidth=1, zorder=2)

# --- tidy legend: keep only one entry per stage and put in lower right like before ---
# create custom handles to ensure consistent order
handles = []
labels = []
for st in stages:
    handles.append(mlines.Line2D([], [], color=palette[st], marker='o', linestyle='None',
                                 markeredgecolor='black', markersize=6))
    labels.append(st)

ax.legend(handles=handles, labels=labels, loc='lower left', handlelength=3)

# # Example of adding a significance bar
# x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
# y, h, col = 31, 1, 'black'
# ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
# ax.text((x1+x2)*.5, y+h+0.2, "***", ha='center', va='bottom', color=col)

# ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [28, 28+h, 28+h, 28], lw=1.5, c=col)
# ax.text((x1+x1)*.5, 28+h+0.2, "n.s.", ha='center', va='bottom', color=col)

# ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [26, 26+h, 26+h, 26], lw=1.5, c=col)
# ax.text((x2+x2)*.5, 26+h+0.2, "n.s.", ha='center', va='bottom', color=col)
plt.title('Acuity of both eyes across sessions', fontsize = 15)
plt.savefig('../Data/Plots/Narrative_plots/fam/cutoff_BE.png', dpi=600, bbox_inches='tight')
plt.figure(figsize=(8, 6))





# Contrast Sensitivity

In [ ]:
# for the test
df = bv_diff['contrast_sensitivity'].copy()
df['group'] = 'test'
indexID = df[ (df['ID'] == 'P01') ].index # | (df['ID'] == 'P02') | (df['ID'] == 'P03')
df.drop(indexID , inplace=True)
df['amb'] = df['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_long = df.melt(id_vars=['ID', 'Staircase', 'amb', 'group'], value_vars=['familiarisation', 'pretraining', 'posttraining'], 
                  var_name='Stage', value_name='Contrast')
df_long['Stage'] = df_long['Stage'].map({'familiarisation':'Familiarisation', 'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})


In [ ]:
df_long_clean = df_long.copy()
df_long_clean_BE = df_long_clean[(df_long_clean['amb'] == 'Both Eyes')]
df_long_clean_BE['amb'] = pd.Categorical(df_long_clean_BE['amb'], categories=['Both Eyes'], ordered=True)
df_long_clean_BE['Sensitivity'] = df_long_clean_BE['Contrast']

df_long_clean = df_long_clean[(df_long_clean['amb'] != 'Both Eyes')]
df_long_clean['amb'] = pd.Categorical(df_long_clean['amb'], categories=['Strong Eye', 'Amblyopic Eye'], ordered=True)
df_long_clean['Sensitivity'] = df_long_clean['Contrast']
print(df_long_clean.head())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from matplotlib.lines import Line2D

# -------------------------
# Prepare data
# -------------------------
# use df_temp as provided by you; filter to test group only
# Define the tick labels (assuming Staircase is 1 to 6)
# sf_labels = ['1/16', '1/8', '1/4', '1/2', '3/4', '1']
# df_long_clean['Staircase'] = df_long_clean['Staircase'].astype(int)  # ensure int for sorting

df = df_long_clean.copy()
df = df[df['group'] == 'test'].reset_index(drop=True)

# map staircase -> spatial frequency
sf_map = {
    1: 1/16,
    2: 1/8,
    3: 1/4,
    4: 1/2,
    5: 3/4,
    6: 1
}
df['SF'] = df['Staircase'].map(sf_map)

# make sure amb values match expected strings; list unique to check if needed:
# print(df['amb'].unique())

# -------------------------
# Aggregate (means + sem)
# -------------------------
agg = (
    df
    .groupby(['amb', 'Stage', 'SF'], as_index=False)
    .agg(
        mean_cs=('Sensitivity', 'mean'),
        sem_cs=('Sensitivity', 'sem')
    )
)

# -------------------------
# Fit log-parabola in log10 space
# -------------------------

fits = {}
for (amb, stage), sub in agg.groupby(['amb', 'Stage']):
    x = sub['SF'].values
    y = np.log10(sub['mean_cs'].values)

    # initial guesses: a ~ max(y), b small positive, f_peak ~ SF at max mean
    if len(x) < 3:
        fits[(amb, stage)] = None
        continue

    p0 = [np.max(y), 0.8, x[np.argmax(y)]]
    try:
        popt, _ = curve_fit(log_parabola, x, y, p0=p0, maxfev=10000)
        fits[(amb, stage)] = popt
    except Exception:
        fits[(amb, stage)] = None

# -------------------------
# Plotting: both amb on same axes
# -------------------------
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(7, 6))

# desired orders and colors
amb_order = ['Strong Eye', 'Amblyopic Eye']  # adjust if your labels differ
stage_order = ['Familiarisation', 'Pre-Training', 'Post-Training']
colors = {'Familiarisation':'#0063D5', 'Pre-Training': '#D55E00', 'Post-Training': '#009E73'}
linestyles = {'Strong Eye': '-', 'Amblyopic Eye': '--'}

# dense SFs for smooth fitted curves
sf_dense = np.logspace(np.log10(1/16), np.log10(1), 400)

# Plot raw points (low alpha) for test only, but color by Stage and shape by amb (optional)
for stage in stage_order:
    sub = df[df['Stage'] == stage]
    ax.scatter(
        sub['SF'],
        sub['Sensitivity'],
        color=colors[stage],
        alpha=0.12,
        s=18,
        zorder=1,
        label=None if stage == 'Pre-Training' else None  # avoid legend duplication here
    )

# Plot aggregated means+SEM and fitted curves for each amb × stage
for amb in amb_order:
    sub_agg = agg[agg['amb'] == amb]
    for stage in stage_order:
        stage_df = sub_agg[sub_agg['Stage'] == stage].sort_values('SF')
        if stage_df.empty:
            continue

        # mean + sem points (markers)
        ax.errorbar(
            stage_df['SF'],
            stage_df['mean_cs'],
            yerr=stage_df['sem_cs'],
            fmt='o',
            color=colors[stage],
            capsize=3,
            zorder=3,
            label=f"{amb} — {stage}"  # label each curve separately (4 labels)
        )

        # fitted curve (if available)
        popt = fits.get((amb, stage))
        if popt is not None:
            y_fit = 10**bv.log_parabola(sf_dense, *popt)
            ax.plot(
                sf_dense,
                y_fit,
                color=colors[stage],
                linestyle=linestyles.get(amb, '-'),
                linewidth=2.5,
                zorder=4
            )

# -------------------------
# Formatting axes (log-log)
# -------------------------
ax.set_xscale('log')
ax.set_yscale('log')

ax.set_xlim(0.055, 1.125)
ax.set_ylim(1, 100)

sf_ticks = np.array([1/16, 1/8, 1/4, 1/2, 3/4, 1])
sf_ticklabels = ['1/16', '1/8', '1/4', '1/2', '3/4', '1']
ax.set_xticks(sf_ticks)
ax.set_xticklabels(sf_ticklabels, fontsize=12)

ax.set_yticks([1, 10, 100])
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

# Labels
ax.set_xlabel('Spatial Frequency (Staircase - Ratio to Cut-Off Threshold)', fontsize=14)
ax.set_ylabel('Contrast Sensitivity (1 / Weber Contrast)', fontsize=14)

# Legend: it will contain four entries (amb × stage)
ax.legend(title='Eye — Stage', fontsize=9, title_fontsize=10, frameon=True, loc='lower left')



legend_handles = [
    # --- Stage section ---
    Line2D([], [], linestyle='none', label='Stage'),
    Line2D([0], [0], color=colors['Familiarisation'], lw=2.5, label='  Familiarisation'),
    Line2D([0], [0], color=colors['Pre-Training'], lw=2.5, label='  Pre-Training'),
    Line2D([0], [0], color=colors['Post-Training'], lw=2.5, label='  Post-Training'),

    # spacer
    Line2D([], [], linestyle='none', label=''),

    # --- Eye section ---
    Line2D([], [], linestyle='none', label='Eye'),
    Line2D([0], [0], color='black', lw=2.5, linestyle='-', label='  Strong Eye'),
    Line2D([0], [0], color='black', lw=2.5, linestyle='--', label='  Amblyopic Eye'),
]

ax.legend(
    handles=legend_handles,
    loc='lower left',
    frameon=True,
    fontsize=10,
    handlelength=3
)


plt.tight_layout()
plt.title('Contrast Sensitivity Curves Across Sessions', fontsize = 15)

plt.savefig('../Data/Plots/Narrative_plots/fam/cs_curves.png', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:

# -------------------------
# Prepare data
# -------------------------
# use df_temp as provided by you; filter to test group only
df = df_long_clean_BE.copy()
df = df[df['group'] == 'test'].reset_index(drop=True)

# map staircase -> spatial frequency
sf_map = {
    1: 1/16,
    2: 1/8,
    3: 1/4,
    4: 1/2,
    5: 3/4,
    6: 1
}
df['SF'] = df['Staircase'].map(sf_map)

# make sure amb values match expected strings; list unique to check if needed:
# print(df['amb'].unique())

# -------------------------
# Aggregate (means + sem)
# -------------------------
agg = (
    df
    .groupby(['amb', 'Stage', 'SF'], as_index=False)
    .agg(
        mean_cs=('Sensitivity', 'mean'),
        sem_cs=('Sensitivity', 'sem')
    )
)

# -------------------------
# Fit log-parabola in log10 space
# -------------------------

fits = {}
for (amb, stage), sub in agg.groupby(['amb', 'Stage']):
    x = sub['SF'].values
    y = np.log10(sub['mean_cs'].values)

    # initial guesses: a ~ max(y), b small positive, f_peak ~ SF at max mean
    if len(x) < 3:
        fits[(amb, stage)] = None
        continue

    p0 = [np.max(y), 0.8, x[np.argmax(y)]]
    try:
        popt, _ = curve_fit(log_parabola, x, y, p0=p0, maxfev=10000)
        fits[(amb, stage)] = popt
    except Exception:
        fits[(amb, stage)] = None

# -------------------------
# Plotting: both amb on same axes
# -------------------------
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(7, 6))

# desired orders and colors
amb_order = ['Both Eyes']  # adjust if your labels differ
stage_order = ['Familiarisation','Pre-Training', 'Post-Training']
colors = {'Familiarisation':'#0063D5','Pre-Training': '#D55E00', 'Post-Training': '#009E73'}
linestyles = {'Both Eyes': '-'}

# dense SFs for smooth fitted curves
sf_dense = np.logspace(np.log10(1/16), np.log10(1), 400)

# Plot raw points (low alpha) for test only, but color by Stage and shape by amb (optional)
for stage in stage_order:
    sub = df[df['Stage'] == stage]
    ax.scatter(
        sub['SF'],
        sub['Sensitivity'],
        color=colors[stage],
        alpha=0.12,
        s=18,
        zorder=1,
        label=None if stage == 'Pre-Training' else None  # avoid legend duplication here
    )

# Plot aggregated means+SEM and fitted curves for each amb × stage
for amb in amb_order:
    sub_agg = agg[agg['amb'] == amb]
    for stage in stage_order:
        stage_df = sub_agg[sub_agg['Stage'] == stage].sort_values('SF')
        if stage_df.empty:
            continue

        # mean + sem points (markers)
        ax.errorbar(
            stage_df['SF'],
            stage_df['mean_cs'],
            yerr=stage_df['sem_cs'],
            fmt='o',
            color=colors[stage],
            capsize=3,
            zorder=3,
            label=f"{amb} — {stage}"  # label each curve separately (4 labels)
        )

        # fitted curve (if available)
        popt = fits.get((amb, stage))
        if popt is not None:
            y_fit = 10**bv.log_parabola(sf_dense, *popt)
            ax.plot(
                sf_dense,
                y_fit,
                color=colors[stage],
                linestyle=linestyles.get(amb, '-'),
                linewidth=2.5,
                zorder=4
            )

# -------------------------
# Formatting axes (log-log)
# -------------------------
ax.set_xscale('log')
ax.set_yscale('log')

ax.set_xlim(0.055, 1.125)
ax.set_ylim(1, 100)

sf_ticks = np.array([1/16, 1/8, 1/4, 1/2, 3/4, 1])
sf_ticklabels = ['1/16', '1/8', '1/4', '1/2', '3/4', '1']
ax.set_xticks(sf_ticks)
ax.set_xticklabels(sf_ticklabels)

ax.set_yticks([1, 10, 100])
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

# Labels
ax.set_xlabel('Spatial Frequency (Staircase - Ratio to Cut-Off Threshold)', fontsize=14)
ax.set_ylabel('Contrast Sensitivity (1 / Weber Contrast)', fontsize=14)

# Legend: it will contain four entries (amb × stage)
ax.legend(title='Both Eyes — Stage', fontsize=9, title_fontsize=10, frameon=True, loc='lower left')



legend_handles = [
    # --- Stage section ---
    Line2D([], [], linestyle='none', label='Both Eyes - Stage'),
    Line2D([0], [0], color=colors['Familiarisation'], lw=2.5, label='  Familiarisation'),
    Line2D([0], [0], color=colors['Pre-Training'], lw=2.5, label='  Pre-Training'),
    Line2D([0], [0], color=colors['Post-Training'], lw=2.5, label='  Post-Training'),

    # # spacer
    # Line2D([], [], linestyle='none', label=''),

    # # --- Eye section ---
    # Line2D([], [], linestyle='none', label='Eye'),
    # Line2D([0], [0], color='black', lw=2.5, linestyle='-', label='  Strong Eye'),
    # Line2D([0], [0], color='black', lw=2.5, linestyle='--', label='  Amblyopic Eye'),
]

ax.legend(
    handles=legend_handles,
    loc='lower left',
    frameon=True,
    fontsize=10,
    handlelength=3
)


plt.tight_layout()
plt.title('Contrast Sensitivity Curves Across Sessions for Both Eyes', fontsize = 15)

plt.savefig('../Data/Plots/Narrative_plots/fam/cs_curves_BE.png',dpi=600, bbox_inches='tight')
plt.show()


In [ ]:

df_long_clean['ID'] = df_long_clean['ID'].astype('category')
df_long_clean['Stage'] = df_long_clean['Stage'].astype('category')
df_long_clean['Staircase'] = df_long_clean['Staircase'].astype('category')

# Run two-way repeated measures ANOVA for test 
aov = pg.rm_anova(
    dv='Sensitivity',
    within=['Stage', 'Staircase'],
    subject='ID',
    data=df_long_clean,
    detailed=True
)

print(aov)

### No AUC done yet - will do if necessary

In [ ]:
bv_auc = pd.read_excel(path_data_ref+'/bv_4_results_ABC_diff.xlsx', sheet_name=None) 


# Interocular Imbalance
The relative contrast needed in each eye for balanced (dichoptic) perception, summarised as logBP.

In [ ]:
df = bv_diff['binoc'].copy()
df['group'] = 'test'
indexID = df[ (df['ID'] == 'P01') ].index #| (df['ID'] == 'P02') | (df['ID'] == 'P03')
df.drop(indexID , inplace=True)
df['amb'] = df['amb'].map({'Both': 'Both Eyes'})
df_long = df.melt(id_vars=['ID', 'Staircase', 'group'], value_vars=['familiarisation','pretraining', 'posttraining'], 
                  var_name='Stage', value_name='logBP')
df_long['Stage'] = df_long['Stage'].map({'familiarisation':'Familiarisation','pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})


In [ ]:

aov = pg.rm_anova(dv='logBP', within=['Stage','Staircase'], subject='ID', data=df_long)
aov.round(3)



In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -----------------------
# Config
# -----------------------
df = df_long.copy()
df = df.dropna()
y_col = 'logBP'

stair_order = [1, 2, 3, 4]
stair_labels = ['0.5', '1', '2', '4']

stage_order = ['Familiarisation', 'Pre-Training', 'Post-Training']
colors = {'Familiarisation':'#0063D5', 'Pre-Training': '#D55E00', 'Post-Training': '#009E73'}

jitter = 0.08
rng = np.random.default_rng(42)

# ensure types
df['Staircase'] = df['Staircase'].astype(int)
df['ID'] = df['ID'].astype(str)
# df_control = df[df['group'] == 'control'].copy()
# df = df[df['group'] == 'test'].copy()

# -----------------------
# Figure
# -----------------------
sns.set_theme(style="whitegrid", palette="pastel")
fig, ax = plt.subplots(figsize=(8, 5.5))

# -----------------------
# Scatter points (test only, alpha=0.3)
# -----------------------
stair_to_x = {s: i for i, s in enumerate(stair_order)}
hue_offsets = {'Familiarisation':-0.15, 'Pre-Training': 0, 'Post-Training': 0.15}

df['x_base'] = df['Staircase'].map(stair_to_x).astype(float)
df['x_offset'] = df['Stage'].map(hue_offsets).astype(float)
df['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df)).astype(float)
df['x_plot'] = df['x_base'] + df['x_offset'] + df['x_jitter']

for stage in stage_order:
    sub = df[df['Stage'] == stage]
    ax.scatter(
        sub['x_plot'],
        sub[y_col],
        color=colors[stage],
        edgecolors='black',
        linewidths=0.6,
        s=40,
        alpha=0.2,
        zorder=3
    )

# -----------------------
# Linear fits ± SD (test population)
# -----------------------
x_dense = np.linspace(0.9, 4.1, 300)

for stage in stage_order:
    sub = df[df['Stage'] == stage]
    if sub.shape[0] < 3:
        continue

    x = sub['Staircase'].values.astype(float)
    y = sub[y_col].values.astype(float)

    # linear fit
    coeffs = np.polyfit(x, y, 1)
    y_fit_dense = np.polyval(coeffs, x_dense)
    y_fit_obs = np.polyval(coeffs, x)

    # SD of residuals
    sd = (y - y_fit_obs).std(ddof=1)

    # plot fit
    ax.plot(
        x_dense - 1,      # map Staircase 1..4 → x positions 0..3
        y_fit_dense,
        color=colors[stage],
        linewidth=2.8,
        zorder=5
    )

    # SD band
    ax.fill_between(
        x_dense - 1,
        y_fit_dense - sd,
        y_fit_dense + sd,
        color=colors[stage],
        alpha=0,
        zorder=4
    )


# Linear fit of controls

# x = df_control['Staircase'].values.astype(float)
# y = df_control[y_col].values.astype(float)

 # linear fit
# coeffs = np.polyfit(x, y, 1)
# y_fit_dense = np.polyval(coeffs, x_dense)
# y_fit_obs = np.polyval(coeffs, x)

# # SD of residuals
# sd = (y - y_fit_obs).std(ddof=1)

# # plot fit
# ax.plot(
#     x_dense - 1,      # map Staircase 1..4 → x positions 0..3
#     y_fit_dense,
#     color='black',
#     linestyle='-',
#     alpha = 0.5,
#     linewidth=2.8,
#     zorder=5
# )

# -----------------------
# Horizontal balance reference line
# -----------------------
ax.axhline(
    y=0.5,
    color='blue',
    linestyle='--',
    linewidth=3,
    alpha=0.9,
    zorder=1
)

# -----------------------
# Axes formatting
# -----------------------
ax.set_xticks(np.arange(len(stair_labels)))
ax.set_xticklabels(stair_labels)
ax.set_xlabel("Spatial Frequency (cycles per dva)", fontsize = 14)
ax.set_ylabel("log of Balance Point (logBP)", fontsize =14)

ymin = 0#np.floor(df[y_col].min() - 0.5)
ymax = 3#np.ceil(df[y_col].max() + 0.5)
ax.set_ylim(ymin, ymax)
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

# -----------------------
# Legend
# -----------------------
legend_handles = [
    Line2D([], [], marker='o', linestyle='none',
           markerfacecolor=colors['Familiarisation'], markeredgecolor='black',
           markersize=8, label='Familiarisation (points)'),
    Line2D([], [], marker='o', linestyle='none',
           markerfacecolor=colors['Pre-Training'], markeredgecolor='black',
           markersize=8, label='Pre-Training (points)'),
    Line2D([], [], marker='o', linestyle='none',
           markerfacecolor=colors['Post-Training'], markeredgecolor='black',
           markersize=8, label='Post-Training (points)'),
    Line2D([0], [0], color=colors['Familiarisation'], lw=2.8, label='Familiarisation'),
    Line2D([0], [0], color=colors['Pre-Training'], lw=2.8, label='Pre-Training (linear fit)'),
    Line2D([0], [0], color=colors['Post-Training'], lw=2.8, label='Post-Training (linear fit)'),
    # Line2D([0], [0], color='black', alpha = 0.5, lw=2.8, label='Controls (linear fit)'),
    Line2D([0], [0], color='blue', lw=3, linestyle='--', label='Balance value (0.5)')
]

ax.legend(handles=legend_handles, loc='upper right', frameon=True, fontsize=10)

plt.tight_layout()
plt.title('Interocular Imbalance Across Sessions', fontsize = 15)

plt.savefig('../Data/Plots/Narrative_plots/fam/logBP_linear_fits.png',dpi=600,bbox_inches='tight')
plt.show()


# Dichoptic Noise
How much noise in the fellow eye it takes to disrupt the amblyopic eye's perception.

In [ ]:
df = bv_diff['dichop'].copy()
df['group'] = 'test'
indexID = df[ (df['ID'] == 'P01') ].index # | (df['ID'] == 'P02') | (df['ID'] == 'P03')
df.drop(indexID , inplace=True)
df['amb'] = df['amb'].map({'Both': 'Both Eyes'})
df_long = df.melt(id_vars=['ID', 'group'], value_vars=['familiarisation','pretraining', 'posttraining'], 
                  var_name='Stage', value_name='Contrast')
df_long['Stage'] = df_long['Stage'].map({'familiarisation':'Familiarisation','pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})


In [ ]:

from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# ---------- Prepare df_long_total and unify value column ----------
df = df_long.copy()
df['Contrast'] = df['Contrast']*100
# possible value columns used earlier: 'Value', 'Contrast', 'Frequency', 'ABC_full', 'logBP'
value_candidates = ['Value', 'Contrast', 'Frequency', 'ABC_full', 'logBP']
value_col = None
for c in value_candidates:
    if c in df.columns:
        value_col = c
        break
if value_col is None:
    # fallback: try to detect numeric column aside from ID, group, Stage, amb
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if numeric_cols:
        value_col = numeric_cols[0]
    else:
        raise ValueError("Couldn't find a value column in df_long_total")

# unify to 'Value'
if value_col != 'Value':
    df = df.rename(columns={value_col: 'Value'})

# # Normalize Stage and group strings
# df['Stage'] = df['Stage'].astype(str).str.replace('-', ' ').str.strip()
# df['Stage'] = df['Stage'].replace({
#     'pretraining': 'Pre-Training',
#     'pre_training': 'Pre-Training',
#     'Pretraining': 'Pre-Training',
#     'posttraining': 'Post-training',
#     'post_training': 'Post-training',
#     'Posttraining': 'Post-training'
# })
df['group'] = df['group'].astype(str).str.lower()  # 'test' / 'control' expected

# keep only the two stages we care about
stage_order = ['Familiarisation', 'Pre-Training', 'Post-Training']
df = df[df['Stage'].isin(stage_order)].copy()

# ensure ID is string
df['ID'] = df['ID'].astype(str)

# split test vs control
df_test = df[df['group'] == 'test'].reset_index(drop=True)
# df_control = df[df['group'] == 'control'].reset_index(drop=True)

# ---------- Compute control mean ± SD per Stage ----------
# ctrl_stats = (
#     df_control
#     .groupby('Stage', as_index=False)['Value']
#     .agg(mean='mean', sd='std', n='count')
# )
# enforce stage ordering
# ctrl_stats['Stage'] = pd.Categorical(ctrl_stats['Stage'], categories=stage_order, ordered=True)
# ctrl_stats = ctrl_stats.sort_values('Stage')

# ---------- Plotting params ----------
sns.set_theme(style="white")
colors = {'Familiarisation':'#0063D5', 'Pre-Training': '#D55E00', 'Post-Training': '#009E73'}
# control_color = '#1E90FF'
box_width = 0.35
range_width = 0.5
jitter = 0.06
rng = np.random.default_rng(42)

# ---------- Make figure ----------
fig, ax = plt.subplots(figsize=(7, 6))

# Draw control mean ± SD rectangles behind everything
stage_to_x = {stage: i for i, stage in enumerate(stage_order)}
# for _, row in ctrl_stats.iterrows():
#     stage = row['Stage']
#     if pd.isna(stage):
#         continue
#     x_center = stage_to_x[stage]
#     mean = row['mean']
#     sd = 0.0 if np.isnan(row['sd']) else float(row['sd'])

    # # mean ± SD rectangle
    # rect_sd = Rectangle(
    #     (x_center - range_width/2, mean - sd),
    #     width=range_width,
    #     height=2 * sd,
    #     facecolor=control_color,
    #     alpha=0.18,
    #     edgecolor=None,
    #     zorder=0
    # )
    # ax.add_patch(rect_sd)

    # # mean horizontal line
    # ax.hlines(
    #     y=mean,
    #     xmin=x_center - range_width/2,
    #     xmax=x_center + range_width/2,
    #     linewidth=3,
    #     color=control_color,
    #     alpha=0.95,
    #     zorder=1
    # )

# ---------- Test-only boxplot ----------
# create ordered categorical for Stage so seaborn places them left->right
df_test['Stage_cat'] = pd.Categorical(df_test['Stage'], categories=stage_order, ordered=True)
sns.boxplot(
    data=df_test,
    x='Stage_cat',
    y='Value',
    showfliers=False,
    width=box_width,
    linewidth=1.5,
    palette={stage_order[0]: colors[stage_order[0]], stage_order[1]: colors[stage_order[1]], stage_order[2]: colors[stage_order[2]]},
    ax=ax,
    zorder=3
)

# ---------- Manual scatter for test points (deterministic dodge + jitter) ----------
df_test = df_test.reset_index(drop=True)
df_test['x_base'] = df_test['Stage'].map(stage_to_x).astype(float)
hue_offsets = {'Familiarisation': -box_width/4, 'Pre-Training': box_width/8,'Post-Training': box_width/4}
df_test['x_offset'] = df_test['Stage'].map(hue_offsets).astype(float)
df_test['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df_test)).astype(float)
df_test['x_plot'] = df_test['x_base'] + df_test['x_offset'] + df_test['x_jitter']

for stage in stage_order:
    sub = df_test[df_test['Stage'] == stage]
    ax.scatter(
        sub['x_plot'],
        sub['Value'],
        color=colors[stage],
        edgecolors='black',
        linewidths=0.8,
        s=40,
        alpha=0.95,
        zorder=6,
        label=None
    )

# ---------- Paired lines connecting Pre -> Post for test subjects ----------
for pid, g in df_test.groupby('ID'):
    if set(stage_order).issubset(g['Stage'].values):
        fam = g[g['Stage'] == 'Familiarisation'].iloc[0]
        pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
        post = g[g['Stage'] == 'Post-Training'].iloc[0]
        ax.plot([fam['x_plot'], pre['x_plot'], post['x_plot']],
                [fam['Value'], pre['Value'], post['Value']],
                color='black', alpha=0.4, linewidth=1, zorder=5)

# ---------- Aesthetics ----------
ax.set_xticks(list(stage_to_x.values()))
ax.set_xticklabels(stage_order)
ax.set_xlabel("")
ax.set_ylabel("Noise Contrast (%)", fontsize=14)  # change label if you'd prefer the original name

# y-limits and ticks: add some padding
ymin = float(df['Value'].min())
ymax = float(df['Value'].max())
yrange = ymax - ymin if (ymax - ymin) != 0 else 1.0
pad = max(0.1 * yrange, 0.1)
ax.set_ylim(0, 110) # ymin - pad*0.15, ymax + pad*0.2
ax.yaxis.set_major_locator(plt.MaxNLocator(7))
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

# ---------- Legend: Stage colours + control entry ----------
legend_handles = [
    Line2D([], [], linestyle='none', label='Stage'),
    Line2D([0], [0], color=colors['Familiarisation'], lw=2.5, label='  Familiarisation'),
    Line2D([0], [0], color=colors['Pre-Training'], lw=2.5, label='  Pre-Training'),
    Line2D([0], [0], color=colors['Post-Training'], lw=2.5, label='  Post-Training')
    # Line2D([], [], linestyle='none', label=''),
    # Line2D([0], [0], color=control_color, lw=2.5, label='Control mean ± SD')
]
ax.legend(handles=legend_handles, loc='upper left', frameon=True, fontsize=10, handlelength=3)

plt.tight_layout()
plt.title('Dichoptic Test Across Sessions', fontsize = 15)
plt.savefig('../Data/Plots/Narrative_plots/fam/noise_contrast_dichoptic.png',dpi=600, bbox_inches='tight')
plt.show()


In [ ]:

aov = pg.rm_anova(dv='Contrast', within=['Stage'], subject='ID', data=df_long)
aov.round(3)



# Asteroid
Stereoacuity (depth-perception threshold), measured in arcseconds of disparity.

In [ ]:
df = bv_diff['asteroid'].copy()
df['group'] = 'test'
indexID = df[ (df['ID'] == 'P01') ].index # | (df['ID'] == 'P02') | (df['ID'] == 'P03')
df.drop(indexID , inplace=True)
df_long = df.melt(id_vars=['ID', 'group'], value_vars=['familiarisation', 'pretraining', 'posttraining'], 
                  var_name='Stage', value_name='log arcseconds')
df_long['Stage'] = df_long['Stage'].map({'familiarisation':'Familiarisation','pretraining': 'Pre-Training', 'posttraining': 'Post-training'})
df_long['arcseconds'] = 10 ** df_long['log arcseconds']


In [ ]:
df = bv_diff['asteroid'].copy()[['ID', 'familiarisation', 'pretraining', 'posttraining', 'Diff']]
indexID = df[ (df['ID'] == 'P01')].index # | (df['ID'] == 'P03')
df.drop(indexID , inplace=True)
df_2 = df.drop('Diff', axis=1)

In [ ]:
cols_to_replace = ['familiarisation', 'pretraining', 'posttraining']
#df = df.dropna()
df_2[cols_to_replace] = df_2[cols_to_replace].replace([3.0], np.nan)
#df_2 = df_2[df_2['pretraining'].notna()]
print(df_2)



In [ ]:
# Melt the dataframe
df_melted = df_2.melt(id_vars='ID', value_vars=['familiarisation', 'pretraining', 'posttraining'],
                    var_name='Stage', value_name='log_arcseconds')
df_melted['Stage'] = df_melted['Stage'].map({'familiarisation':'Familiarisation','pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})
df_melted['arcseconds'] = 10**df_melted['log_arcseconds'] # to go back in arcseconds


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Ellipse

# IDs you want to exclude from the *post-training* box stats
outlier_ids = ['P08', 'P10', 'P17', 'P19', 'P21']

# --- 1) Draw the boxplots from a filtered dataframe (stats ignore the 3 post-training points) ---
df_for_boxes = df_melted[
    ~(
        (df_melted['Stage'] == 'Post-Training') &
        (df_melted['ID'].isin(outlier_ids))
     )
]

plt.figure(figsize=(4, 6))
sns.boxplot(
    data=df_for_boxes,
    x='Stage',
    y='log_arcseconds',
    # y = 'arcseconds',
    order=['Familiarisation','Pre-Training', 'Post-Training'],   # lock the positions: 0 = Pre, 1 = Post
    palette={'Familiarisation':'#0063D5', 'Post-Training': '#009E73', 'Pre-Training': '#D55E00'}, width=0.5, fill=False
)

plt.axhline(y=2.6, color='blue', zorder=1)

plt.axhline(y=1.778, color='pink', zorder=1)
plt.axhline(y=1.477, color='pink', zorder=1)
plt.axhspan(1.477, 1.778, color='pink', alpha=0.15, zorder=0)

# plt.axhline(y=60, color='pink', zorder=1)
# plt.axhline(y=30, color='pink', zorder=1)
# plt.axhspan(60, 30, color='pink', alpha=0.15, zorder=0)

# --- 2) Overlay subject labels, highlights, and connecting lines from the *full* dataframe ---
grouped = df_melted.groupby('ID')
jitter = 0.25

# Optional: make jitter reproducible
rng = np.random.default_rng(42)

for ID, group in grouped:
    x_jittered = {}

    for _, row in group.iterrows():
        if (row['Stage'] == 'Familiarisation'):
            x_pos = 0 
        elif (row['Stage'] == 'Pre-Training'):
            x_pos = 1
        else:
            x_pos = 2

        jittered_x = x_pos + rng.uniform(-jitter, jitter)
        x_jittered[row['Stage']] = jittered_x

        # If this is one of the 3 special IDs on the post-training side, draw a filled ellipse
        if (row['Stage'] == 'Post-Training') and (ID in outlier_ids):
            circle = Ellipse(
                (jittered_x, row['log_arcseconds']),
                width=0.24, height=0.1,
                edgecolor='black', facecolor='#d4c8ec',
                linewidth=0.5, zorder=3
            )
            plt.gca().add_patch(circle)

        # Label each point (draw on top)
        plt.text(
            jittered_x, row['log_arcseconds'],str(ID),
            ha='center', va='center', zorder=5, fontsize=12
        )


    # Connect pre ↔ post if both exist
    if len(group) >= 2 and 'Pre-Training' in x_jittered and 'Post-Training' in x_jittered:
        y_fam = group[group['Stage'] == 'Familiarisation']['log_arcseconds'].values[0]
        y_pre = group[group['Stage'] == 'Pre-Training']['log_arcseconds'].values[0]
        y_post = group[group['Stage'] == 'Post-Training']['log_arcseconds'].values[0]

    #     # Shorten the line a bit by nudging x toward the center
        x_fam = x_jittered['Familiarisation'] + 0.12
        x_pre = x_jittered['Pre-Training']
        x_post = x_jittered['Post-Training'] - 0.12

        plt.plot(
            [x_fam, x_pre, x_post],
            [y_fam, y_pre, y_post],
            color='black', alpha=0.3, linewidth=1, zorder=2
        )

plt.text(0.5, 1.55, 'Normal Range', color='red', alpha=0.5, style = 'italic', horizontalalignment='center', verticalalignment='center')

# plt.text(0.5, 3, 'Clinical Threshold', color='blue', alpha=0.5, style = 'italic', horizontalalignment='center', verticalalignment='center')

# Example significance bar (adjust y to your data range)
# x1, x2 = 0, 1
# y, h, col = 3, 0.05, 'black'
# plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
# plt.text((x1+x2)*.5, y+h+0.02, "*", ha='center', va='bottom', color=col)

# Final formatting
# plt.title('Stereoacuity', fontsize=18)
plt.ylim(1.25, 3.2)

plt.ylabel('log arcseconds', fontsize=14)
plt.xlabel('')
plt.grid(True, axis='y', linestyle='--', alpha=0.3)
plt.xticks([0, 1, 2], ['Familiarisation', 'Pre-Training', 'Post-Training'])  # matches 'order=' above
plt.tight_layout()
plt.title('Asteroid Test Across Sessions', fontsize=15)
plt.savefig('../Data/Plots/Narrative_plots/fam/stereoacuity.png', dpi=600, bbox_inches='tight')
plt.show()


# Crowded Tumbling E
Visual acuity for a Tumbling-E optotype surrounded by flanking contours (crowding).

In [ ]:
df = bv_diff['crowded'].copy()
df['group'] = 'test'
indexID = df[ (df['ID'] == 'P01') ].index # | (df['ID'] == 'P02') | (df['ID'] == 'P03')
df.drop(indexID , inplace=True)
df['amb'] = df['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_long = df.melt(id_vars=['ID', 'amb', 'group'], value_vars=['familiarisation', 'pretraining', 'posttraining'], 
                  var_name='Stage', value_name='logMAR')
df_long['Stage'] = df_long['Stage'].map({'familiarisation':'Familiarisation','pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})


In [ ]:
df_long_clean = df_long.copy()
df_long_clean_BE = df_long_clean[(df_long_clean['amb'] == 'Both Eyes')]
df_long_clean_BE['amb'] = pd.Categorical(df_long_clean_BE['amb'], categories=['Both Eyes'], ordered=True)

df_long_clean = df_long_clean[(df_long_clean['amb'] != 'Both Eyes')]
df_long_clean['amb'] = pd.Categorical(df_long_clean['amb'], categories=['Strong Eye', 'Amblyopic Eye'], ordered=True)


In [ ]:


import pingouin as pg 

aov = pg.rm_anova(dv='logMAR', within=['amb','Stage'], subject='ID', data=df_long_clean)
aov.round(3)

In [ ]:

# Set a better style
sns.set_theme(style="white", palette="pastel")

# Set up the plot
plt.figure(figsize=(7, 6))

# Create the boxplot with colors for each amb condition
ax = sns.boxplot(
    x='amb', y='logMAR', data=df_long_clean, hue='Stage', showfliers=False, width=0.6, linewidth=1.5, 
    #palette={'Lazy Eye': '#66CDAA', 'Both Eyes': '#1E90FF', 'Strong Eye': '#FF6347'}
    palette={'Familiarisation':'#0063D5','Post-Training': '#009E73', 'Pre-Training': '#D55E00'}
)

strip = sns.stripplot(data=df_long_clean, x="amb", y="logMAR", hue='Stage', dodge=True, ax=ax, 
                palette={'Familiarisation':'#0063D5','Post-Training': '#009E73', 'Pre-Training': '#D55E00'}, edgecolor='black', linewidth=1)

#ax.set_yticks(range(0, 1))
# ax.set_yticklabels([0, 0.2, 0.4, 0.6, 0.8, 1, 1.2, 1.4, 1.6, 1.8])
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=[(handles[0]), (handles[1]), (handles[2])],
          labels=['Familiarisation', 'Pre-Training', 'Post-Training'],
          loc='center left', handlelength=3,
          handler_map={tuple: HandlerTuple(ndivide=None)})

# Map amb categories to numerical positions for plotting
amb_order = df['amb'].unique()
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# Customize axis labels
ax.set_ylabel("log of the Maximum Angle Resolution (logMAR)", fontsize=14)  # or whatever label you'd like
ax.set_xlabel("")  # removes the label but keeps 'Strong Eye' / 'Lazy Eye' ticks

# Get cleaned categorical order for amb
amb_order = df_long_clean['amb'].cat.categories
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# === Extract real x/y points from stripplot ===
# Make sure the order of the DataFrame matches how seaborn plots the points
# Seaborn plots in order: hue nested inside x, so sort by 'amb' then 'Stage'
df_sorted = df_long_clean.sort_values(by=['amb', 'Stage'], key=lambda col: col.astype(str))
df_sorted = df_sorted.dropna()
# Get all stripplot points
offsets = []
for coll in strip.collections:
    offsets.extend(coll.get_offsets())

# Map points to their (ID, amb) and Stage
coord_map = {}  # {(ID, amb): {stage: (x, y)}}
for i, (_, row) in enumerate(df_sorted.iterrows()):
    x, y = offsets[i]
    key = (row['ID'], row['amb'])
    stage = row['Stage']
    if key not in coord_map:
        coord_map[key] = {}
    coord_map[key][stage] = (x, y)

# Plot lines for each pair of Pre/Post
for key, stages in coord_map.items():
    if len(stages) == 3 and 'Familiarisation' in stages and 'Pre-Training' in stages and 'Post-Training' in stages:
        x_vals = [stages['Familiarisation'][0], stages['Pre-Training'][0], stages['Post-Training'][0]]
        y_vals = [stages['Familiarisation'][1], stages['Pre-Training'][1], stages['Post-Training'][1]]
        ax.plot(x_vals, y_vals, color='black', alpha=0.4, linewidth=1)
    elif len(stages) == 2 and 'Pre-Training' in stages and 'Post-Training' in stages:
        x_vals = [stages['Pre-Training'][0], stages['Post-Training'][0]]
        y_vals = [stages['Pre-Training'][1], stages['Post-Training'][1]]
        ax.plot(x_vals, y_vals, color='black', alpha=0.4, linewidth=1)

# Example of adding a significance bar
x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
y, h, col = 1.7, 0.05, 'black'
ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
ax.text((x1+x2)*.5, y+h+0.005, "***", ha='center', va='bottom', color=col)
# y =0.8
# ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [y, y+h, y+h, y], lw=1.5, c=col)
# ax.text((x1+x1)*.5, y+h+0.02, "n.s.", ha='center', va='bottom', color=col)
# y = 1.5
# ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [y, y+h, y+h, y], lw=1.5, c=col)
# ax.text((x2+x2)*.5, y+h+0.02, "n.s.", ha='center', va='bottom', color=col)


plt.ylim(-0.05,1.85)
plt.title('Crowded Acuity Across Sessions', fontsize=15)
plt.savefig('../Data/Plots/Narrative_plots/fam/Tumbling_crowded.png', dpi=600, bbox_inches='tight')
plt.show()




In [ ]:


# ---------- Input df ----------
# expects df with columns: ID, amb, group ('test'/'control'), Stage ('Pre-Training'/'Post-training'), logMAR
df_vis = df_long_clean_BE.copy()
df_vis['ID'] = df_vis['ID'].astype(str)
df_vis['Stage'] = df_vis['Stage'].astype(str)

# ---------- Split test / control ----------
df_test = df_vis[df_vis['group'].str.lower() == 'test'].copy().reset_index(drop=True)
df_control = df_vis[df_vis['group'].str.lower() == 'control'].copy().reset_index(drop=True)

# ensure amb ordering (use categorical if not already)
if not pd.api.types.is_categorical_dtype(df_vis['amb']):
    amb_order = sorted(df_vis['amb'].unique(), key=str)
    df_vis['amb'] = pd.Categorical(df_vis['amb'], categories=amb_order, ordered=True)
    df_test['amb'] = pd.Categorical(df_test['amb'], categories=amb_order, ordered=True)
    df_control['amb'] = pd.Categorical(df_control['amb'], categories=amb_order, ordered=True)
else:
    amb_order = list(df_vis['amb'].cat.categories)

stage_order = ['Familiarisation', 'Pre-Training', 'Post-Training']

# ---------- Plot style ----------
sns.set_theme(style="white")
test_palette = {'Familiarisation':'#0063D5','Pre-Training': '#D55E00', 'Post-Training': '#009E73'}

# ---------- Figure ----------
fig, ax = plt.subplots(figsize=(7, 6))

box_width = 0.6
range_width = 0.36
offsets = {'Familiarisation': -box_width / 3, 'Pre-Training': 0, 'Post-Training': box_width / 3}

# ---------- Test boxplots ----------
sns.boxplot(
    data=df_test,
    x='amb',
    y='logMAR',
    hue='Stage',
    order=amb_order,
    hue_order=stage_order,
    width=box_width,
    palette=test_palette,
    linewidth=1.5,
    showfliers=False,
    ax=ax,
    zorder=2
)

# ---------- Manual test dots (deterministic jitter) ----------
jitter = 0.06
rng = np.random.default_rng(42)
df_test = df_test.reset_index(drop=True)
df_test['x_base'] = df_test['amb'].map({a: i for i, a in enumerate(amb_order)}).astype(float)
df_test['x_offset'] = df_test['Stage'].map(offsets).astype(float)
df_test['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df_test)).astype(float)
df_test['x_plot'] = df_test['x_base'] + df_test['x_offset'] + df_test['x_jitter']

for stage in stage_order:
    sub = df_test[df_test['Stage'] == stage]
    ax.scatter(
        sub['x_plot'],
        sub['logMAR'],
        color=test_palette[stage],
        edgecolors='black',
        linewidths=0.8,
        s=40,
        alpha=0.95,
        zorder=4,
        label=stage
    )

# # ---------- Connect pre ↔ post per test subject & amb ----------
# for (pid, amb), g in df_test.groupby(['ID', 'amb']):
#     if set(stage_order).issubset(g['Stage'].values):
#         pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
#         post = g[g['Stage'] == 'Post-training'].iloc[0]
#         ax.plot(
#             [pre['x_plot'], post['x_plot']],
#             [pre['logMAR'], post['logMAR']],
#             color='black',
#             alpha=0.4,
#             linewidth=1,
#             zorder=3
#         )

# ---------- Axes / Legend / Save ----------
ax.set_ylabel("log of the Maximum Angle Resolution (logMAR)", fontsize=14)  # or whatever label you'd like
ax.set_xlabel("")
# y-limits and ticks: choose sensible padding from data
ymin = float(df_vis['logMAR'].min())
ymax = float(df_vis['logMAR'].max())
yrange = ymax - ymin if ymax != ymin else 0.5
pad = max(0.1 * yrange, 0.1)
ax.set_ylim(ymin - pad*0.15, ymax + pad*0.2)

ax.set_xticks(range(len(amb_order)))
ax.set_xticklabels(amb_order)

# example significance annotation (edit positions/labels)
# x1, x2 = 0, 1
# y, h, col = ax.get_ylim()[1] - pad*0.05, pad*0.03, 'black'
# ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
# ax.text((x1+x2)*.5, y+h+0.01, "***", ha='center', va='bottom', color=col)

# clean legend: keep one entry per Stage
handles, labels = ax.get_legend_handles_labels()
uniq = dict(zip(labels, handles))
ax.legend(
    [uniq[k] for k in stage_order if k in uniq],
    [k for k in stage_order if k in uniq],
    loc='upper left',
    handlelength=3
)

plt.tight_layout()
plt.title('Crowded Acuity Across Sessions for Both Eyes', fontsize = 15)
plt.savefig('../Data/Plots/Narrative_plots/fam/logMAR_cutoff_BE.png', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:


import pingouin as pg 

aov = pg.rm_anova(dv='logMAR', within=['Stage'], subject='ID', data=df_long_clean_BE)
aov.round(3)

# Tumbling E Acuity
Uncrowded visual acuity for the Tumbling-E optotype.

In [ ]:
df = bv_diff['snellen'].copy()
df['amb'] = df['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
indexID = df[ (df['ID'] == 'P01') ].index #| (df['ID'] == 'P02') | (df['ID'] == 'P03')| (df['ID'] == 'P07')
df.drop(indexID , inplace=True)

df_long = df.copy().melt(id_vars=['ID', 'amb'], value_vars=['familiarisation', 'pretraining', 'posttraining'], 
                  var_name='Stage', value_name='logMAR')
df_long['Stage'] = df_long['Stage'].map({'familiarisation':'Familiarisation', 'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})



In [ ]:
df_long_clean = df_long.copy()
df_long_clean_BE = df_long_clean[(df_long_clean['amb'] == 'Both Eyes')]
df_long_clean_BE['amb'] = pd.Categorical(df_long_clean_BE['amb'], categories=['Both Eyes'], ordered=True)

df_long_clean = df_long_clean[(df_long_clean['amb'] != 'Both Eyes')]
df_long_clean['amb'] = pd.Categorical(df_long_clean['amb'], categories=['Strong Eye', 'Amblyopic Eye'], ordered=True)


In [ ]:

import pingouin as pg 

aov = pg.rm_anova(dv='logMAR', within=['amb','Stage'], subject='ID', data=df_long_clean)
aov.round(3)

In [ ]:

# Set a better style
sns.set_theme(style="white", palette="pastel")

# Set up the plot
plt.figure(figsize=(7, 6))

# Create the boxplot with colors for each amb condition
ax = sns.boxplot(
    x='amb', y='logMAR', data=df_long_clean, hue='Stage', showfliers=False, width=0.6, linewidth=1.5, 
    #palette={'Lazy Eye': '#66CDAA', 'Both Eyes': '#1E90FF', 'Strong Eye': '#FF6347'}
    palette={'Familiarisation':'#0063D5','Post-Training': '#009E73', 'Pre-Training': '#D55E00'}
)

strip = sns.stripplot(data=df_long_clean, x="amb", y="logMAR", hue='Stage', dodge=True, ax=ax, 
                palette={'Familiarisation':'#0063D5','Post-Training': '#009E73', 'Pre-Training': '#D55E00'}, edgecolor='black', linewidth=1)

#ax.set_yticks(range(0, 1))
# ax.set_yticklabels([0, 0.2, 0.4, 0.6, 0.8, 1, 1.2, 1.4, 1.6, 1.8])
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=[(handles[0]), (handles[1]), (handles[2])],
          labels=['Familiarisation', 'Pre-Training', 'Post-Training'],
          loc='center left', handlelength=3,
          handler_map={tuple: HandlerTuple(ndivide=None)})

# Map amb categories to numerical positions for plotting
amb_order = df['amb'].unique()
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# Customize axis labels
ax.set_ylabel("log of the Maximum Angle Resolution (logMAR)", fontsize=14)  # or whatever label you'd like
ax.set_xlabel("")  # removes the label but keeps 'Strong Eye' / 'Lazy Eye' ticks

# Get cleaned categorical order for amb
amb_order = df_long_clean['amb'].cat.categories
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# === Extract real x/y points from stripplot ===
# Make sure the order of the DataFrame matches how seaborn plots the points
# Seaborn plots in order: hue nested inside x, so sort by 'amb' then 'Stage'
df_sorted = df_long_clean.sort_values(by=['amb', 'Stage'], key=lambda col: col.astype(str))
df_sorted = df_sorted.dropna()
# Get all stripplot points
offsets = []
for coll in strip.collections:
    offsets.extend(coll.get_offsets())

# Map points to their (ID, amb) and Stage
coord_map = {}  # {(ID, amb): {stage: (x, y)}}
for i, (_, row) in enumerate(df_sorted.iterrows()):
    x, y = offsets[i]
    key = (row['ID'], row['amb'])
    stage = row['Stage']
    if key not in coord_map:
        coord_map[key] = {}
    coord_map[key][stage] = (x, y)

# Plot lines for each pair of Pre/Post
for key, stages in coord_map.items():
    if len(stages) == 3 and 'Familiarisation' in stages and 'Pre-Training' in stages and 'Post-Training' in stages:
        x_vals = [stages['Familiarisation'][0], stages['Pre-Training'][0], stages['Post-Training'][0]]
        y_vals = [stages['Familiarisation'][1], stages['Pre-Training'][1], stages['Post-Training'][1]]
        ax.plot(x_vals, y_vals, color='black', alpha=0.4, linewidth=1)
    elif len(stages) == 2 and 'Pre-Training' in stages and 'Post-Training' in stages:
        x_vals = [stages['Pre-Training'][0], stages['Post-Training'][0]]
        y_vals = [stages['Pre-Training'][1], stages['Post-Training'][1]]
        ax.plot(x_vals, y_vals, color='black', alpha=0.4, linewidth=1)

# Example of adding a significance bar
x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
y, h, col = 1.7, 0.05, 'black'
ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
ax.text((x1+x2)*.5, y+h+0.005, "***", ha='center', va='bottom', color=col)
# y =0.8
# ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [y, y+h, y+h, y], lw=1.5, c=col)
# ax.text((x1+x1)*.5, y+h+0.02, "n.s.", ha='center', va='bottom', color=col)
# y = 1.5
# ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [y, y+h, y+h, y], lw=1.5, c=col)
# ax.text((x2+x2)*.5, y+h+0.02, "n.s.", ha='center', va='bottom', color=col)


# plt.ylim(-0.05,1.85)
plt.title('Snellen Acuity Across Sessions', fontsize=15)
plt.savefig('../Data/Plots/Narrative_plots/fam/Tumbling_E.png', dpi=600, bbox_inches='tight')
plt.show()


